# 9장. 회귀 분석으로 숫자 예측하기

이 노트북은 `book/chapters/ch09_regression_analysis.md`와 `src/regression.py`의 회귀 분석 흐름을 그대로 실행합니다.

핵심은 높은 성능을 만드는 것이 아니라 다음 원칙을 지키는 것입니다.

- 예측 시점을 먼저 정의합니다.
- 주문 상세의 수량·단가·금액은 목표값 생성에만 사용합니다.
- 식별자와 예측 이후 정보를 입력값에서 제외합니다.
- 날짜 순서로 훈련·테스트 데이터를 나눕니다.
- 단순 평균 베이스라인보다 실제로 나은지 확인합니다.
- 낮거나 음수인 R²도 숨기지 않고 해석합니다.


## 0. 실행 전 확인

이 노트북은 5장에서 만든 다음 파일을 사용합니다.

- `data/processed/customers_clean.csv`
- `data/processed/orders_clean.csv`
- `data/processed/order_items_clean.csv`

파일이 없다면 프로젝트 루트에서 다음 명령을 먼저 실행합니다.

```bash
python scripts/preprocess_data.py
```


## 1. 회귀 문제와 예측 시점

예측 대상은 주문별 주문 상세 금액 합계인 `order_total`입니다.

교육용 예측 시점에서는 주문 시점과 결제 수단, 고객의 비식별 특성은 알 수 있지만 주문 상세의 수량·단가·금액은 모델에 제공하지 않는다고 가정합니다.

따라서 `quantity`, `unit_price`, `line_total`, `item_count`, `total_quantity`, `avg_unit_price`, `order_status`, `order_id`, `customer_id`는 입력값으로 사용하지 않습니다.


## 2. 패키지와 프로젝트 경로 설정


In [2]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

from src.regression import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_leakage_checklist,
    build_regression_dataset,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print("프로젝트 루트:", PROJECT_ROOT.resolve())
print("전처리 데이터:", PROCESSED_DIR.resolve())
print("보고서 폴더:", REPORT_DIR.resolve())


프로젝트 루트: C:\dev\llm-data-analysis-study\chapter09
전처리 데이터: C:\dev\llm-data-analysis-study\chapter09\data\processed
보고서 폴더: C:\dev\llm-data-analysis-study\chapter09\reports


## 3. 전처리 데이터 불러오기

원본 CSV로 자동 대체하지 않습니다. 5장의 전처리 기준과 다른 데이터가 섞이면 실습 결과를 비교하기 어렵기 때문입니다.


In [3]:
data = load_regression_source_data(PROCESSED_DIR)

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
orders: (300, 7)
order_items: (764, 6)


## 4. 누수 없는 주문 단위 모델링 데이터 만들기

`order_total`은 주문 상세에서 계산하지만, 주문 상세에서 만든 다른 집계값은 모델 입력에 포함하지 않습니다.


In [4]:
model_data = build_regression_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

print("모델링 데이터:", model_data.shape)
display(
    model_data[
        [
            "order_id",
            "order_date",
            *FEATURE_COLUMNS,
            TARGET_COLUMN,
        ]
    ].head()
)


모델링 데이터: (300, 11)


,order_id,order_date,order_month,order_dayofweek,age,payment_method,gender,city,order_total
0,238,2025-09-18,9,3,67,bank_transfer,M,대구,361000
1,79,2025-09-20,9,5,63,bank_transfer,F,광주,283000
2,12,2025-09-21,9,6,61,kakao_pay,F,인천,444000
3,169,2025-09-21,9,6,69,kakao_pay,M,서울,448000
4,152,2025-09-22,9,0,63,naver_pay,F,광주,985000


In [5]:
print("숫자형 입력값:", NUMERIC_FEATURES)
print("범주형 입력값:", CATEGORICAL_FEATURES)
print("예측 대상:", TARGET_COLUMN)
print("금지 입력값:", sorted(FORBIDDEN_FEATURES))

validate_feature_columns(FEATURE_COLUMNS)


숫자형 입력값: ['order_month', 'order_dayofweek', 'age']
범주형 입력값: ['payment_method', 'gender', 'city']
예측 대상: order_total
금지 입력값: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'quantity', 'total_quantity', 'unit_price']


## 5. 날짜 순서로 훈련·테스트 데이터 분할

과거 주문으로 이후 주문을 예측하는 상황을 모방하기 위해 무작위 분할 대신 날짜 순서 분할을 사용합니다.


In [6]:
train_data, test_data = split_model_data_by_time(
    model_data,
    test_size=0.2,
)

X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()

print(
    "훈련 기간:",
    train_data["order_date"].min(),
    "~",
    train_data["order_date"].max(),
)
print(
    "테스트 기간:",
    test_data["order_date"].min(),
    "~",
    test_data["order_date"].max(),
)
print("훈련:", X_train.shape, "테스트:", X_test.shape)


훈련 기간: 2025-09-18 00:00:00 ~ 2026-06-18 00:00:00
테스트 기간: 2026-06-18 00:00:00 ~ 2026-09-17 00:00:00
훈련: (240, 6) 테스트: (60, 6)


## 6. 베이스라인·선형 회귀·랜덤 포레스트 비교

각 모델의 전처리 파이프라인은 훈련 데이터 안에서만 결측치 대체, 표준화, 원-핫 인코딩을 학습합니다.

`Baseline Mean`은 훈련 목표값의 평균을 모든 테스트 주문에 예측합니다. 다른 모델은 최소한 이 기준과 비교해야 합니다.


In [7]:
models, model_comparison, predictions = (
    train_and_evaluate_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        random_state=42,
    )
)

model_comparison


,model,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,412811.979167,455895.833333,531015.673566,-0.002361,0.00
1,Linear Regression,394909.155289,480838.770125,563389.260829,-0.128305,-5.47
2,Random Forest,326283.984977,481964.281371,560735.685832,-0.117701,-5.72


`MAE_improvement_vs_baseline_pct`가 양수이면 베이스라인보다 평균 절대 오차가 줄었습니다. 음수이면 복잡한 모델이 단순 평균보다도 나쁘다는 뜻입니다.

훈련 MAE가 매우 낮고 테스트 MAE가 크면 과적합 가능성을 확인합니다.


## 7. 시간 순서 교차검증

한 번의 테스트 기간 결과만 믿지 않고, 훈련 기간 안에서 여러 시점으로 나누어 성능의 평균과 변동을 확인합니다.


In [8]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
    max_splits=5,
)

cv_summary


,model,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Random Forest,424718.425469,10916.685653,-0.122760
1,Linear Regression,486206.815570,86092.951085,-0.529589


교차검증 MAE 표준편차가 크면 기간에 따라 성능이 불안정할 수 있습니다. R²가 반복적으로 음수라면 현재 입력값으로 평균 예측을 안정적으로 넘지 못할 가능성이 큽니다.


## 8. 실제값·예측값과 잔차 확인

테스트 MAE가 가장 낮은 비베이스라인 모델을 진단용으로 선택합니다. 선택된 모델이 베이스라인보다 낫다는 뜻은 아니므로 비교표를 함께 확인해야 합니다.


In [9]:
selected_model_name = select_diagnostic_model(
    model_comparison
)

prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)

print("진단 모델:", selected_model_name)
prediction_result.head(10)


진단 모델: Linear Regression


,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-08-25,1877000,6.063998e+05,1.270600e+06,1.270600e+06,Linear Regression
1,168,2026-08-21,10000,9.802708e+05,-9.702708e+05,9.702708e+05,Linear Regression
2,179,2026-09-17,86000,9.929754e+05,-9.069754e+05,9.069754e+05,Linear Regression
3,96,2026-07-21,1690000,7.866559e+05,9.033441e+05,9.033441e+05,Linear Regression
4,73,2026-08-24,72000,9.721201e+05,-9.001201e+05,9.001201e+05,Linear Regression
5,183,2026-08-02,1685000,7.946286e+05,8.903714e+05,8.903714e+05,Linear Regression
6,21,2026-08-29,1767000,8.851689e+05,8.818311e+05,8.818311e+05,Linear Regression
7,153,2026-06-18,172000,1.034316e+06,-8.623162e+05,8.623162e+05,Linear Regression
8,15,2026-08-02,1677000,8.148517e+05,8.621483e+05,8.621483e+05,Linear Regression
9,248,2026-06-18,55000,9.111336e+05,-8.561336e+05,8.561336e+05,Linear Regression


In [11]:
prediction_result["abs_error"].describe()


count    6.000000e+01
mean     4.808388e+05
std      2.960787e+05
min      3.153171e+04
25%      2.168984e+05
50%      4.720138e+05
75%      7.079347e+05
max      1.270600e+06
Name: abs_error, dtype: float64

`residual = 실제값 - 예측값`입니다.

- 잔차가 양수이면 실제값을 낮게 예측했습니다.
- 잔차가 음수이면 실제값을 높게 예측했습니다.
- 큰 절대 오차가 특정 기간이나 금액대에 몰리는지 확인합니다.


In [12]:
# 3번 항목 검증 셀 (읽기 전용)
print("order_items 컬럼:", order_items.columns.tolist())
print("orders 컬럼:", orders.columns.tolist())
print()

# 1. line_total 검증
calc = order_items["quantity"] * order_items["unit_price"]
print("[line_total 검증]")
print("불일치 행:", (calc != order_items["line_total"]).sum())
print("결측:", order_items["line_total"].isna().sum())
print("음수:", (order_items["line_total"] < 0).sum())
print()

# 2. orders와 target 관계 검증
target_check = order_items.groupby("order_id")["line_total"].sum().rename("order_total_calc")
print("[orders와 target 관계 검증]")
print("합산 결과 행 수:", len(target_check))
print("orders의 order_id 중복:", orders["order_id"].duplicated().sum())
print("orders에만 있는 order_id:", len(set(orders["order_id"]) - set(target_check.index)))
print("order_items에만 있는 order_id:", len(set(target_check.index) - set(orders["order_id"])))

# model_data의 order_total과 직접 합산한 값 비교
md = model_data.set_index("order_id")["order_total"]
diff = (md - target_check.reindex(md.index)).abs()
print("model_data order_total과 재계산 값의 최대 차이:", diff.max())
print()

# 3. orders와 customers 관계 검증
print("[orders와 customers 관계 검증]")
print("customers의 customer_id 중복:", customers["customer_id"].duplicated().sum())
print("customers에 없는 customer_id를 가진 주문:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
check_merged = orders.merge(customers, on="customer_id", how="left", validate="m:1")
print("병합 후 행 수:", len(check_merged))
print("model_data 행 수:", len(model_data))

order_items 컬럼: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
orders 컬럼: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[line_total 검증]
불일치 행: 0
결측: 0
음수: 0

[orders와 target 관계 검증]
합산 결과 행 수: 300
orders의 order_id 중복: 0
orders에만 있는 order_id: 0
order_items에만 있는 order_id: 0
model_data order_total과 재계산 값의 최대 차이: 0

[orders와 customers 관계 검증]
customers의 customer_id 중복: 0
customers에 없는 customer_id를 가진 주문: 0
병합 후 행 수: 300
model_data 행 수: 300


## 9. 체크리스트와 결과 저장

예측 결과에는 주문 식별자가 포함되므로 `_internal` 파일은 외부 공개용으로 사용하지 않습니다.


In [13]:
regression_checklist = build_leakage_checklist()
regression_checklist


,check_item,status
0,예측 시점이 명확한가?,□
1,목표값과 목표값의 계산 재료를 입력에서 제외했는가?,□
2,예측 이후에 알 수 있는 정보를 사용하지 않았는가?,□
3,식별자를 일반 숫자 변수로 사용하지 않았는가?,□
4,전처리기가 훈련 데이터 안에서만 학습되는가?,□
5,시간 순서 또는 업무 목적에 맞는 분할을 사용했는가?,□
6,단순 베이스라인과 비교했는가?,□
7,테스트 데이터로 최종 성능을 평가했는가?,□
8,"MAE, RMSE, R²를 올바르게 해석했는가?",□
9,음수 R²와 낮은 성능을 숨기지 않았는가?,□


In [14]:
output_paths = save_regression_outputs(
    model_data=model_data,
    train_data=train_data,
    test_data=test_data,
    model_comparison=model_comparison,
    cv_summary=cv_summary,
    prediction_result=prediction_result,
    checklist=regression_checklist,
    selected_model_name=selected_model_name,
    report_dir=REPORT_DIR,
)

for name, path in output_paths.items():
    print(f"{name}: {path}")


model_data_internal: c:\dev\llm-data-analysis-study\chapter09\reports\ch09_regression_model_data_internal.csv
model_comparison: c:\dev\llm-data-analysis-study\chapter09\reports\ch09_regression_model_comparison.csv
cv_summary: c:\dev\llm-data-analysis-study\chapter09\reports\ch09_regression_cv_summary.csv
predictions_internal: c:\dev\llm-data-analysis-study\chapter09\reports\ch09_regression_predictions_internal.csv
checklist: c:\dev\llm-data-analysis-study\chapter09\reports\ch09_regression_checklist.csv
report: c:\dev\llm-data-analysis-study\chapter09\reports\ch09_regression_report.md
actual_vs_predicted_figure: c:\dev\llm-data-analysis-study\chapter09\reports\figures\ch09_actual_vs_predicted.png
residual_figure: c:\dev\llm-data-analysis-study\chapter09\reports\figures\ch09_residual_histogram.png


In [35]:
import pandas as pd

selected = "Random Forest"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. split_summary
train_dt = pd.to_datetime(train_data["order_date"])
test_dt = pd.to_datetime(test_data["order_date"])
overlap_dates = sorted(set(train_dt.dt.date) & set(test_dt.dt.date))

split_rows = [
    ("train_start", train_dt.min().date()),
    ("train_end", train_dt.max().date()),
    ("train_rows", len(train_data)),
    ("test_start", test_dt.min().date()),
    ("test_end", test_dt.max().date()),
    ("test_rows", len(test_data)),
    ("overlap_date_count", len(overlap_dates)),
    ("overlap_dates", ";".join(str(d) for d in overlap_dates)),
    ("boundary_train_rows", int(train_dt.dt.date.isin(overlap_dates).sum())),
    ("boundary_test_rows", int(test_dt.dt.date.isin(overlap_dates).sum())),
]
split_summary = pd.DataFrame(split_rows, columns=["item", "value"])

# 2. feature_audit
audit_rows = [
    ("order_month", "O", "X", "X", "사용", "주문 시점에 확정된 달력 정보다. 훈련 기간에 7월과 8월이 없어 분포 불일치 위험이 있다"),
    ("order_dayofweek", "O", "X", "X", "사용", "주문 시점에 확정된 달력 정보다"),
    ("age", "O", "X", "X", "사용", "주문 전부터 존재하는 고객 속성이다"),
    ("gender", "O", "X", "X", "사용", "주문 전부터 존재하는 고객 속성이다"),
    ("city", "O", "X", "X", "사용", "주문 전부터 존재하는 고객 속성이다"),
    ("payment_method", "O", "X", "X", "사용", "예측 시점을 결제 수단 선택 시점으로 정의해서 사용한다. 제외 실험에서 MAE가 줄어 성능을 부풀린 흔적은 없다"),
    ("order_date", "O", "X", "X", "제외", "입력으로 쓰지 않고 분할 기준과 월 및 요일 생성에만 사용한다"),
    ("order_total", "X", "O", "X", "제외", "Target 자체다"),
    ("line_total", "X", "O", "X", "제외", "주문 상세의 금액이며 합산하면 order_total이 된다"),
    ("unit_price", "X", "O", "X", "제외", "주문 상세의 단가로 order_total을 만드는 재료다"),
    ("quantity", "X", "O", "X", "제외", "주문 상세의 수량으로 order_total을 만드는 재료다"),
    ("total_quantity", "X", "O", "X", "제외", "주문 상세에서 집계한 값으로 order_total과 직접 연결된다"),
    ("item_count", "X", "O", "X", "제외", "주문 상세에서 집계한 값으로 order_total과 직접 연결된다"),
    ("avg_unit_price", "X", "O", "X", "제외", "주문 상세에서 집계한 값으로 Target 계산 재료에 해당한다"),
    ("order_status", "X", "X", "O", "제외", "주문 이후에 확정되는 처리 결과다"),
    ("order_id", "X", "X", "O", "제외", "식별자다. 예측 정보가 없고 개별 주문 암기로만 작동한다"),
    ("customer_id", "X", "X", "O", "제외", "식별자다. 고객 단위 암기로 이어질 수 있다"),
]
feature_audit = pd.DataFrame(
    audit_rows,
    columns=["feature", "usable_at_prediction", "target_ingredient",
             "post_event_or_id", "final_use", "reason"],
)

# 3. validation
overlap_cnt = len(set(FEATURE_COLUMNS) & set(FORBIDDEN_FEATURES))
strict_ok = bool(train_data["order_date"].max() < test_data["order_date"].min())
sel_in_cv = selected in cv_summary["model"].values
base_in_test = "Baseline Mean" in model_comparison["model"].values
sel_in_test = selected in model_comparison["model"].values
n_test = len(y_test)

validation = pd.DataFrame([
    ("forbidden_feature_overlap", overlap_cnt, "PASS" if overlap_cnt == 0 else "FAIL"),
    ("strict_train_before_test", strict_ok, "PASS" if strict_ok else "FAIL"),
    ("selected_model_exists_in_train_cv", sel_in_cv, "PASS" if sel_in_cv else "FAIL"),
    ("final_test_contains_baseline", base_in_test, "PASS" if base_in_test else "FAIL"),
    ("final_test_contains_frozen_selected_model", sel_in_test, "PASS" if sel_in_test else "FAIL"),
    ("test_rows_for_r2", n_test, "PASS" if n_test >= 2 else "FAIL"),
], columns=["check", "value", "status"])

# 저장
split_summary.to_csv(REPORT_DIR / "ch09_regression_split_summary.csv", index=False, encoding="utf-8-sig")
feature_audit.to_csv(REPORT_DIR / "ch09_regression_feature_audit.csv", index=False, encoding="utf-8-sig")
validation.to_csv(REPORT_DIR / "ch09_regression_validation.csv", index=False, encoding="utf-8-sig")

print(split_summary.to_string(index=False))
print()
print(validation.to_string(index=False))
print()
print("feature_audit 행 수:", len(feature_audit))
print(sorted(p.name for p in REPORT_DIR.glob("*.csv")))

               item      value
        train_start 2025-09-18
          train_end 2026-06-18
         train_rows        240
         test_start 2026-06-18
           test_end 2026-09-17
          test_rows         60
 overlap_date_count          1
      overlap_dates 2026-06-18
boundary_train_rows          1
 boundary_test_rows          2

                                    check value status
                forbidden_feature_overlap     0   PASS
                 strict_train_before_test False   FAIL
        selected_model_exists_in_train_cv  True   PASS
             final_test_contains_baseline  True   PASS
final_test_contains_frozen_selected_model  True   PASS
                         test_rows_for_r2    60   PASS

feature_audit 행 수: 17
['ch09_regression_checklist.csv', 'ch09_regression_cv_summary.csv', 'ch09_regression_feature_audit.csv', 'ch09_regression_model_comparison.csv', 'ch09_regression_model_data_internal.csv', 'ch09_regression_predictions_internal.csv', 'ch09_regression_s

저장되는 주요 결과는 다음과 같습니다.

- 모델 비교표
- 시간 순서 교차검증 요약
- 내부용 모델링 데이터와 예측 오차
- 회귀 분석 체크리스트와 Markdown 보고서
- 실제값·예측값 산점도
- 잔차 히스토그램


## 10. 공통 모듈로 전체 파이프라인 다시 실행하기

앞의 단계는 `src/regression.py`의 `run_regression_analysis()`로 한 번에 재현할 수 있습니다.


In [15]:
regression_result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)

regression_result["model_comparison"]


,model,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,412811.979167,455895.833333,531015.673566,-0.002361,0.00
1,Linear Regression,394909.155289,480838.770125,563389.260829,-0.128305,-5.47
2,Random Forest,326283.984977,481964.281371,560735.685832,-0.117701,-5.72


## 11. 실행 스크립트 사용

프로젝트 루트에서 다음 명령을 실행하면 같은 파이프라인이 수행됩니다.

```bash
python scripts/run_regression_analysis.py
```


## 12. LLM에게 회귀 코드를 요청하는 프롬프트

```text
온라인 쇼핑몰 주문 금액을 예측하는 교육용 회귀 모델을 만들려고 합니다.

예측 대상:
- order_total: 주문별 line_total 합계

예측 시점:
- 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만
  주문 상세의 수량, 단가, 금액은 모델에 제공하지 않음

사용 가능한 입력값:
- payment_method
- order_month
- order_dayofweek
- gender
- age
- city

사용하면 안 되는 입력값:
- order_total, line_total
- quantity, unit_price
- item_count, total_quantity, avg_unit_price
- order_status
- order_id, customer_id

요청:
1. 날짜 순서로 훈련 데이터 80%, 테스트 데이터 20%를 나누어 주세요.
2. 결측치 처리와 OneHotEncoder를 Pipeline 안에 넣어 주세요.
3. DummyRegressor, LinearRegression, RandomForestRegressor를 비교해 주세요.
4. MAE, RMSE, R²와 베이스라인 대비 개선율을 계산해 주세요.
5. 훈련·테스트 성능 차이와 데이터 누수 가능성을 설명해 주세요.
6. R²가 음수일 때의 의미도 설명해 주세요.

주의:
- 실제 데이터에 없는 컬럼을 만들지 마세요.
- 테스트 데이터로 전처리 규칙을 학습하지 마세요.
- 높은 성능을 가정하거나 결과를 임의로 만들어내지 마세요.
```


## 13. 실습 과제

1. `payment_method`를 제외했을 때 베이스라인 대비 성능이 어떻게 달라지는지 비교합니다.
2. `RandomForestRegressor`의 `min_samples_leaf` 값을 바꾸어 훈련·테스트 MAE를 비교합니다.
3. 시간 순서 교차검증에서 성능이 가장 낮은 기간의 데이터 특성을 확인합니다.
4. 예측 시점 이전의 고객 구매 이력을 설계하되 미래 정보가 섞이지 않도록 집계 기준일을 명시합니다.
5. LLM이 제안한 회귀 코드에서 목표값의 계산 재료나 식별자가 입력에 포함됐는지 검토합니다.


In [16]:
# 과제 1. FEATURE_COLUMNS에서 payment_method를 제외한 별도 실험을 설계해 보세요.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

MODEL_PARAMS = dict(random_state=42, n_jobs=-1)  

def evaluate(drop_cols=None):
    drop_cols = drop_cols or []
    Xtr = X_train.drop(columns=drop_cols)
    Xte = X_test.drop(columns=drop_cols)
    cat_cols = Xtr.select_dtypes(exclude="number").columns.tolist()

    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
        remainder="passthrough",
    )
    model = Pipeline([("prep", prep), ("rf", RandomForestRegressor(**MODEL_PARAMS))])
    model.fit(Xtr, y_train)
    pred = model.predict(Xte)

    return {
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }

results = pd.DataFrame({
    "baseline": evaluate(),
    "payment_method 제외": evaluate(["payment_method"]),
}).T
results["MAE 변화"] = results["MAE"] - results.loc["baseline", "MAE"]
print(results.round(3))

                          MAE        RMSE     R2     MAE 변화
baseline           500251.544  584877.853 -0.216      0.000
payment_method 제외  486468.583  575961.453 -0.179 -13782.961


### 과제 1. payment_method 제외 실험 설계

가설: `payment_method`는 주문 금액 예측에 기여하지 않으며, 제외해도 성능이 유지되거나 개선된다.

설계: `FEATURE_COLUMNS`에서 `payment_method`만 제거한 `EXPERIMENT_FEATURES`를 만들고, 모델(`RandomForestRegressor`), 하이퍼파라미터, `random_state=42`, 학습/테스트 분할, 전처리를 모두 동일하게 유지한 채 피처 목록만 바꿔 비교한다. 평가 지표는 MAE(주), RMSE, R²다.

결과: 제외 후 MAE는 500,252에서 486,469로 13,783(약 2.8%) 감소했고, RMSE는 584,878에서 575,961로, R²는 -0.216에서 -0.179로 소폭 개선됐다.

해석: `payment_method`는 예측에 기여하지 못했고 노이즈로 작용했을 가능성이 있다. 다만 두 모델 모두 R²가 음수여서 신호가 약한 데이터이며, 차이가 작아 시드를 바꾼 반복 실험으로 일관성을 확인해야 한다.

In [17]:
# 과제 2. RandomForestRegressor의 min_samples_leaf 값을 바꾸어 비교해 보세요.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["payment_method", "gender", "city"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough",
)

for leaf in [1, 3, 5, 10, 20]:
    model = Pipeline([
        ("prep", preprocessor),
        ("rf", RandomForestRegressor(min_samples_leaf=leaf, random_state=42, n_jobs=-1)),
    ])
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    print(leaf, round(train_mae, 2), round(test_mae, 2))

1 164204.94 500251.54
3 275121.82 499143.15
5 325253.73 485861.65
10 368606.12 479987.04
20 393992.34 464494.08


### 과제 2 결과 해석

`min_samples_leaf`를 1, 3, 5, 10, 20으로 바꾸며 훈련/테스트 MAE를 비교했다.

- 값이 커질수록 훈련 MAE는 164205에서 393992로 증가했다. 잎 노드에 최소 표본 수를 요구하면 모델이 훈련 데이터를 외우기 어려워지기 때문이다.
- 반대로 테스트 MAE는 500252에서 464494로 감소했다.
- 훈련-테스트 격차도 336047에서 70502로 줄었다. `min_samples_leaf=1`에서는 훈련 데이터에만 맞춘 과적합이 뚜렷했고, 규제를 강하게 걸수록 일반화 성능이 좋아졌다.
- 이번 후보 중에서는 `min_samples_leaf=20`이 가장 좋았다. 다만 20에서도 테스트 MAE가 계속 감소하는 추세라 30, 50 등 더 큰 값을 탐색할 필요가 있다.
- 훈련 MAE도 20에서 39만 원 수준으로 높아, 현재 입력 변수만으로 설명할 수 있는 신호가 제한적이다.

In [18]:
print("FEATURE_COLUMNS:", FEATURE_COLUMNS)
print()
print("X_train 컬럼:", X_train.columns.tolist())
print()
print("y_train 이름:", getattr(y_train, "name", None))
print()
print("날짜 후보 컬럼:", [c for c in X_train.columns if "date" in c.lower() or "dt" in c.lower() or "time" in c.lower()])
print()
print(X_train.dtypes)

FEATURE_COLUMNS: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']

X_train 컬럼: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']

y_train 이름: order_total

날짜 후보 컬럼: []

order_month        int32
order_dayofweek    int32
age                int64
payment_method       str
gender               str
city                 str
dtype: object


In [19]:
print("X_train 인덱스 예시:", X_train.index[:5].tolist())

# 노트북에서 주문 데이터를 담은 변수 찾기
import pandas as pd
print([n for n, o in globals().items() if isinstance(o, pd.DataFrame) and not n.startswith("_")])

X_train 인덱스 예시: [0, 1, 2, 3, 4]
['customers', 'orders', 'order_items', 'model_data', 'train_data', 'test_data', 'X_train', 'X_test', 'model_comparison', 'cv_summary', 'prediction_result', 'check_merged', 'regression_checklist', 'results']


In [20]:
print("model_data:", model_data.columns.tolist())
print("train_data:", train_data.columns.tolist())
print("model_data 인덱스:", model_data.index[:5].tolist(), len(model_data))
print("test_data 인덱스:", test_data.index[:5].tolist(), len(test_data))
print()
print(cv_summary)   # 앞에서 이미 만든 교차검증 결과일 수 있음

model_data: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek', 'order_total', 'gender', 'age', 'city']
train_data: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek', 'order_total', 'gender', 'age', 'city']
model_data 인덱스: [0, 1, 2, 3, 4] 300
test_data 인덱스: [240, 241, 242, 243, 244] 60

               model    cv_MAE_mean    cv_MAE_std  cv_R2_mean
0      Random Forest  424718.425469  10916.685653   -0.122760
1  Linear Regression  486206.815570  86092.951085   -0.529589


In [21]:
# 과제 3. 시간 순서 교차검증에서 성능이 가장 낮은 기간의 데이터 특성을 확인합니다.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

DATE_COL = "order_date"
TARGET_COL = "order_total"

ts_df = model_data.copy()
ts_df[DATE_COL] = pd.to_datetime(ts_df[DATE_COL])
ts_df = ts_df.sort_values(DATE_COL).reset_index(drop=True)

X_all, y_all = ts_df[FEATURE_COLUMNS], ts_df[TARGET_COL]
cat_cols = X_all.select_dtypes(exclude="number").columns.tolist()

def make_model():
    prep = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
        remainder="passthrough",
    )
    return Pipeline([("prep", prep),
                     ("rf", RandomForestRegressor(random_state=42, n_jobs=-1))])

tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(X_all))
rows = []
for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
    model = make_model().fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx])
    pred = model.predict(X_all.iloc[va_idx])
    va = ts_df.iloc[va_idx]
    mae = mean_absolute_error(y_all.iloc[va_idx], pred)
    rows.append({
        "fold": fold,
        "훈련 행수": len(tr_idx),
        "검증 시작": va[DATE_COL].min().date(),
        "검증 종료": va[DATE_COL].max().date(),
        "MAE": mae,
        "검증 평균금액": va[TARGET_COL].mean(),
        "검증 금액 표준편차": va[TARGET_COL].std(),
        "상대오차(MAE/평균)": mae / va[TARGET_COL].mean(),
    })

cv_result = pd.DataFrame(rows)
print(cv_result.round(2))

worst_fold = int(cv_result.loc[cv_result["MAE"].idxmax(), "fold"])
print("\n최저 성능 fold:", worst_fold)

va_idx = splits[worst_fold - 1][1]
worst_df = ts_df.iloc[va_idx]
others_df = ts_df.drop(ts_df.index[va_idx])

print(pd.DataFrame({
    "최저 fold 검증 기간": worst_df[TARGET_COL].describe(),
    "그 외 전체": others_df[TARGET_COL].describe(),
}).round(1))

print("\n[주문 수] 최저 fold:", len(worst_df), "/ 그 외:", len(others_df))

for col in ["payment_method", "city", "gender"]:
    print(f"\n[{col} 비중]")
    print(pd.DataFrame({
        "최저 fold": worst_df[col].value_counts(normalize=True),
        "그 외": others_df[col].value_counts(normalize=True),
    }).round(3))

   fold  훈련 행수       검증 시작       검증 종료        MAE   검증 평균금액  검증 금액 표준편차  \
0     1     50  2025-11-26  2026-01-10  545209.55  960320.0   532511.20   
1     2    100  2026-01-10  2026-03-12  456294.00  883260.0   500413.29   
2     3    150  2026-03-14  2026-05-04  443162.63  789200.0   490431.46   
3     4    200  2026-05-05  2026-07-01  456151.29  831140.0   511523.69   
4     5    250  2026-07-02  2026-09-17  482112.62  838640.0   545134.16   

   상대오차(MAE/평균)  
0          0.57  
1          0.52  
2          0.56  
3          0.55  
4          0.57  

최저 성능 fold: 1
       최저 fold 검증 기간     그 외 전체
count           50.0      250.0
mean        960320.0   830376.0
std         532511.2   510933.1
min          20000.0    10000.0
25%         545500.0   411250.0
50%         895000.0   774000.0
75%        1295000.0  1115500.0
max        2118000.0  2821000.0

[주문 수] 최저 fold: 50 / 그 외: 250

[payment_method 비중]
                최저 fold    그 외
payment_method                
bank_transfer      0.28 

### 과제 3. 시간 순서 교차검증 결과

`TimeSeriesSplit(n_splits=5)`로 주문일 순서대로 검증했다.

| fold | 검증 기간 | 훈련 행수 | MAE | 상대오차 |
|---|---|---|---|---|
| 1 | 2025-11-26 ~ 2026-01-10 | 50 | 545,210 | 0.57 |
| 2 | 2026-01-10 ~ 2026-03-12 | 100 | 456,294 | 0.52 |
| 3 | 2026-03-14 ~ 2026-05-04 | 150 | 443,163 | 0.56 |
| 4 | 2026-05-05 ~ 2026-07-01 | 200 | 456,151 | 0.55 |
| 5 | 2026-07-02 ~ 2026-09-17 | 250 | 482,113 | 0.57 |

- 성능이 가장 낮은 구간은 fold 1(2025-11-26 ~ 2026-01-10)로 MAE는 545,210이었다.
- 이 기간은 평균 주문 금액이 960320으로 나머지 기간(830376)보다 약 16% 높았고, card 결제 비중이 낮고(16% vs 24.8%) 서울 주문이 없었다는 점이 달랐다.
- 다만 fold 1은 훈련 데이터가 50건으로 가장 적고, 금액 규모를 보정한 상대오차(0.57)는 fold 5와 같아서 특별히 나쁜 구간이라고 보기 어렵다. 따라서 주된 원인은 기간 특성보다 훈련 데이터 부족과 금액 규모 차이로 판단한다.
- 검증 표본이 fold당 50건이라 fold 간 차이는 우연일 수 있어 경향 수준으로 해석한다.

In [22]:
# 과제 4. 예측 시점 이전의 고객 구매 이력을 설계하되 미래 정보가 섞이지 않도록 집계 기준일을 명시합니다.
import pandas as pd

hist_df = model_data.copy()
hist_df["order_date"] = pd.to_datetime(hist_df["order_date"])
hist_df = hist_df.sort_values(["order_date", "order_id"]).reset_index(drop=True)

g = hist_df.groupby("customer_id")

# 과거 주문 횟수: 현재 주문 이전에 몇 번 주문했는가
hist_df["prev_order_count"] = g.cumcount()

# 과거 평균 주문 금액: 현재 주문 금액은 shift로 제외
hist_df["prev_avg_total"] = (
    g["order_total"].transform(lambda s: s.shift(1).expanding().mean())
)

# 마지막 주문 이후 경과일
hist_df["days_since_last_order"] = (
    hist_df["order_date"] - g["order_date"].shift(1)
).dt.days

print(hist_df[["customer_id", "order_date", "order_total",
               "prev_order_count", "prev_avg_total", "days_since_last_order"]].head(10))

   customer_id order_date  order_total  prev_order_count  prev_avg_total  \
0          124 2025-09-18       361000                 0             NaN   
1           62 2025-09-20       283000                 0             NaN   
2           98 2025-09-21       444000                 0             NaN   
3            9 2025-09-21       448000                 0             NaN   
4           62 2025-09-22       985000                 1        283000.0   
5           11 2025-09-23       276000                 0             NaN   
6            4 2025-09-28       603000                 0             NaN   
7           37 2025-09-29       794000                 0             NaN   
8           77 2025-10-02       756000                 0             NaN   
9          114 2025-10-04       690000                 0             NaN   

   days_since_last_order  
0                    NaN  
1                    NaN  
2                    NaN  
3                    NaN  
4                    2.0  
5

In [23]:
has_hist = (hist_df["prev_order_count"] > 0)
print("이력 있는 주문:", has_hist.sum(), "/", len(hist_df), f"({has_hist.mean():.1%})")
print(hist_df["prev_order_count"].value_counts().sort_index())

이력 있는 주문: 174 / 300 (58.0%)
prev_order_count
0    126
1     89
2     51
3     22
4      8
5      3
6      1
Name: count, dtype: int64


### 과제 4. 예측 시점 이전의 구매 이력 설계

집계 기준일: 각 주문의 `order_date`를 예측 시점으로 보고, 같은 고객(`customer_id`)의 `order_date` 이전 주문만 집계한다. 당일과 미래 주문은 포함하지 않으며, 집계 기간은 기준일 이전의 전체 이력이다.

설계한 피처

| 피처 | 의미 | 이력이 없을 때 |
|---|---|---|
| `prev_order_count` | 해당 주문 이전 고객의 주문 횟수 | 0 |
| `prev_avg_total` | 해당 주문 이전 주문 금액의 평균 | 결측 |
| `days_since_last_order` | 직전 주문 이후 경과일 | 결측 |
| `has_history` | 과거 이력 유무 지시 변수 | 0 |

누수 방지 방법
- 주문일 순으로 정렬한 뒤 고객별로 `shift(1)`을 적용해 현재 주문의 금액이 자기 피처에 들어가지 않게 했다.
- 고객 전체 기간의 평균 금액이나 총 주문 횟수는 미래 주문을 포함하므로 사용하지 않는다.
- 학습/테스트 분할 시에도 이력은 시간순으로 계산하므로, 테스트 기간 주문이 훈련 주문의 피처에 반영되지 않는다.

검증 결과
- 고객 62번의 두 번째 주문(9/22)은 `prev_order_count=1`, `prev_avg_total=283,000`, `days_since_last_order=2`로, 직전 주문(9/20)의 금액만 반영되었고 현재 주문 금액(985,000)은 들어가지 않았다.
- 첫 주문의 `prev_avg_total`과 `days_since_last_order`는 모두 결측이다.

한계: 전체 300건 중 이력이 있는 주문은 174건(58.0%)이고 126건(42.0%)은 첫 주문이다. 이력이 있는 주문도 과거 주문이 1~2건인 경우가 대부분이라 평균 금액의 신뢰도가 낮다. 결측은 0 또는 -1로 대체하고 `has_history`와 함께 사용한다.

In [24]:
# 과제 5. LLM이 제안한 회귀 코드에서 목표값의 계산 재료나 식별자가 입력에 포함됐는지 검토합니다.
TARGET_COL = "order_total"

# 목표값 계산 재료 후보 (order_items 기반 컬럼 + 금액 관련 이름)
TARGET_INGREDIENTS = ["quantity", "unit_price", "line_total", "item_total",
                      "order_total", "total_amount", "price", "amount"]
# 식별자 후보
IDENTIFIERS = ["order_id", "customer_id", "product_id", "order_item_id"]
# 그 밖에 입력으로 부적절한 컬럼 (결과 정보)
OUTCOME_COLS = ["order_status"]

def audit_features(feature_cols, label="FEATURE_COLUMNS"):
    ingredient_hits = [c for c in feature_cols if c in TARGET_INGREDIENTS]
    id_hits = [c for c in feature_cols if c in IDENTIFIERS or c.endswith("_id")]
    outcome_hits = [c for c in feature_cols if c in OUTCOME_COLS]
    print(f"[{label}] {feature_cols}")
    print(" - 목표값 계산 재료 포함:", ingredient_hits or "없음")
    print(" - 식별자 포함:", id_hits or "없음")
    print(" - 결과 정보 포함:", outcome_hits or "없음")
    return not (ingredient_hits or id_hits or outcome_hits)

ok = audit_features(FEATURE_COLUMNS)
print("\n점검 통과:", ok)

# 이과제 4에서 만든 이력 피처를 추가할 때도 같은 점검
new_features = FEATURE_COLUMNS + ["prev_order_count", "prev_avg_total",
                                  "days_since_last_order", "has_history"]
audit_features(new_features, "FEATURE_COLUMNS + 이력 피처")

# 데이터프레임에 있지만 X에서 빠진 컬럼도 확인 (제외 사유를 적기 위함)
excluded = [c for c in model_data.columns if c not in FEATURE_COLUMNS]
print("\nX에서 제외된 컬럼:", excluded)

[FEATURE_COLUMNS] ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
 - 목표값 계산 재료 포함: 없음
 - 식별자 포함: 없음
 - 결과 정보 포함: 없음

점검 통과: True
[FEATURE_COLUMNS + 이력 피처] ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city', 'prev_order_count', 'prev_avg_total', 'days_since_last_order', 'has_history']
 - 목표값 계산 재료 포함: 없음
 - 식별자 포함: 없음
 - 결과 정보 포함: 없음

X에서 제외된 컬럼: ['order_id', 'customer_id', 'order_date', 'order_status', 'order_total']


In [25]:
num = model_data.select_dtypes("number")
print(num.corr()["order_total"].sort_values(ascending=False))

order_total        1.000000
order_dayofweek    0.089064
order_month        0.014051
customer_id       -0.000286
age               -0.026194
order_id          -0.064673
Name: order_total, dtype: float64


### 과제 5. LLM 제안 코드 검토 (누수·식별자 점검)

점검 기준
1. 목표값(`order_total`)의 계산 재료(`quantity`, `unit_price`, 금액 합계류)가 입력에 있는가
2. 식별자(`order_id`, `customer_id` 등)가 입력에 있는가
3. 주문 이후에야 알 수 있는 결과 정보(`order_status`)가 입력에 있는가

점검 결과
- 현재 `FEATURE_COLUMNS`(`order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city`)는 세 항목 모두 해당 없음으로 점검을 통과했다.
- 과제 4의 이력 피처(`prev_order_count`, `prev_avg_total`, `days_since_last_order`, `has_history`)를 추가해도 통과했다. 이 피처들은 `shift(1)`로 현재 주문 금액을 제외하고 이전 주문만 집계한다.
- X에서 제외한 컬럼과 사유: `order_id`·`customer_id`(식별자), `order_date`(월·요일 파생 컬럼으로 대체), `order_status`(주문 결과 정보), `order_total`(타깃).

LLM 코드를 받았을 때의 체크리스트
- `X = df.drop(columns=[target])`처럼 타깃만 빼고 전부 쓰는 코드는 식별자와 계산 재료가 따라 들어가기 쉬우므로, 사용할 피처를 `FEATURE_COLUMNS`로 명시한다.
- `customer_id`를 원-핫 인코딩하면 컬럼이 급증하고 과적합이 생긴다.
- 고객/상품 단위 집계는 예측 시점 이전 데이터만 쓰는지 확인한다.

한계: 이 점검은 컬럼 이름 기반이므로 이름에 드러나지 않는 누수는 타깃과의 상관 확인으로 보완해야 한다. 점검은 컬럼 이름 기반이므로, 숫자형 컬럼과 `order_total`의 상관을 추가로 확인했다. 가장 큰 상관은 `order_dayofweek`(0.089)였고 나머지는 절댓값 0.07 이하여서, 목표값 계산 재료가 숨어 있다는 징후는 없었다. 

결론: 현재 입력에는 누수 요소가 확인되지 않았다. 낮은 예측력(R² 음수)은 누수가 아니라 입력 변수가 주문 금액을 설명할 신호가 약하기 때문으로 해석한다.

## 14. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 예측 시점 정의
- 목표값과 목표값 계산 재료 분리
- 식별자와 예측 이후 정보 제외
- 날짜 순서 훈련·테스트 분할
- Pipeline 안에서 결측치 처리와 인코딩
- 평균 베이스라인, 선형 회귀, 랜덤 포레스트 비교
- MAE, RMSE, R²와 베이스라인 대비 개선율 해석
- 시간 순서 교차검증
- 실제값·예측값과 잔차 진단
- 내부 식별 정보 보호
- `src/regression.py`와 실행 스크립트를 통한 재현

낮은 성능도 현재 데이터와 입력 변수의 한계를 알려 주는 유효한 분석 결과입니다.


# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 황지원
- GitHub ID: Jiwon0712
- 작성일: 2026-11-04
- 최종 Notebook URL: https://github.com/Jiwon0712/llm-data-analysis-study/blob/main//chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- Target: order_total (주문 금액, 회귀)
- 예측 단위: 주문 1건
- 예측 시점: 고객이 결제 수단을 선택하는 시점 (상품 구성과 수량이 확정되기 전)
- 예측 시점에 알 수 있는 정보:
  - 주문 시점: order_month, order_dayofweek
  - 고객 속성: age, gender, city
  - 결제 정보: payment_method
- 예측 시점에 알 수 없는 정보:
  - 주문 구성 정보: item_count, quantity, total_quantity, unit_price,
    avg_unit_price, line_total 
  - 주문 처리 결과: order_status 
  - 식별자: order_id, customer_id 
  - Target 자체: order_total

### 나의 해석과 판단
주문 금액은 담은 상품의 수량과 단가로 결정되므로, 이 정보가 확정되기 전인
결제 수단 선택 시점을 예측 시점으로 잡았다. 이 시점에는 주문 월과 요일,
고객 속성(나이, 성별, 도시), 결제 수단만 알 수 있다고 가정했다.
order_items 기반 변수는 order_total을 거의 그대로 계산할 수 있어 누수가 되므로 금지 입력값으로 분리했고, order_status는 주문 이후에 정해지는 사후 정보라 제외했다.

2번의 시간 분할(훈련 2025-09-18 ~ 2026-06-18, 테스트 2026-06-18 ~ 2026-09-17)과 예측 시점이 일관되도록, 모든 입력은 주문 시점에 이미 존재하는 값만 사용했다. 다만 order_month는 훈련 기간(9월부터 6월)에 7월과 8월이 없어 테스트 기간과 분포가 어긋날 수 있으므로, 분포를 확인한 뒤 유지 여부를 결정한다. 이 예측 시점은 학습용 가정이며, 실제 서비스에서는 결제 수단 선택 시점에 장바구니가 이미 확정되어 있을 수 있다.

---

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- | --- |
| order_month | O | X | X | 사용 | 주문 시점에 확정된 달력 정보. 훈련 기간에 7월과 8월이 없어 테스트와 분포가 어긋날 위험이 있음 |
| order_dayofweek | O | X | X | 사용 | 주문 시점에 확정된 달력 정보 |
| age | O | X | X | 사용 | 주문 전부터 존재하는 고객 속성. customers의 현재 값이면 주문 시점 값과 다를 수 있어 생성 방식 확인 필요 |
| gender | O | X | X | 사용 | 주문 전부터 존재하는 고객 속성 |
| city | O | X | X | 사용 | 주문 전부터 존재하는 고객 속성. 주문 이후 이사로 값이 바뀐 경우가 없는지 확인 필요 |
| payment_method | O | X | X | 사용 | 예측 시점을 결제 수단 선택 시점으로 정의해서 사용. 장바구니 확정 후 선택되는 서비스라면 금액이 이미 알려진 시점이라 사후 정보가 됨 |
| order_date | O | X | X | 제외 | 입력으로 쓰지 않고 시간 분할 기준과 month, dayofweek 생성에만 사용 |
| order_total | X | O | X | 제외 | Target 자체 |
| line_total | X | O | X | 제외 | 상품별 금액이며 주문 단위로 합산하면 order_total이 됨 |
| unit_price | X | O | X | 제외 | 수량과 곱해 order_total을 만드는 재료 |
| quantity | X | O | X | 제외 | 단가와 곱해 order_total을 만드는 재료 |
| total_quantity | X | O | X | 제외 | 주문 내 수량 합계로 order_total과 직접 연결됨 |
| item_count | X | O | X | 제외 | 주문 내 상품 수로 order_total과 직접 연결됨 |
| avg_unit_price | X | O | X | 제외 | 단가에서 파생된 값으로 order_total 계산에 쓰이는 재료. 정확한 정의는 코드에서 확인 필요 |
| order_status | X | X | O | 제외 | 주문 이후에 확정되는 처리 결과 |
| order_id | X | X | O | 제외 | 식별자. 예측 정보가 없고 개별 주문을 암기하는 용도로만 작동함 |
| customer_id | X | X | O | 제외 | 식별자. 고객 단위 암기로 이어질 수 있음 |

- 금지 feature overlap 개수: 0 
- 가장 위험하다고 판단한 feature: payment_method
- 이유: 금지 목록에 없는 입력 중 예측 시점 가정에 가장 크게 의존하는 변수이다. 결제 수단이 장바구니 확정 후에 정해지는 서비스라면 그 시점엔 금액도 이미 알려져 있어서 사후 정보가 된다. 고액 주문에 특정 결제 수단이 몰리는 패턴이 있으면 간접 누수가 될 수도 있다. 다만 테스트 MAE가 베이스라인 455896보다 선형회귀 480839와 랜덤포레스트 481964로 오히려 나빠서, 누수가 성능을 부풀린 흔적은 확인되지 않았다. 차순위 위험은 order_month로, 누수는 아니지만 훈련과 테스트의 월 분포 불일치 문제가 있다.

![Leakage 점검](images/step02_leakage.png)

---

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity * unit_price
order_total = order_items에서 order_id별 line_total의 합계
```

- line_total 검증 결과: order_items 764행에서 quantity * unit_price와 line_total이 다른 행은 0건이었다. line_total의 결측은 0건, 음수 값은 0건이었다.
- orders ↔ target 관계 검증 결과: order_items를 order_id별로 합산한 결과는 300행이었다. orders 300행과 비교했을 때 orders에만 있는 order_id는 0건, order_items에만 있는 order_id는 0건이었다. orders의 order_id 중복은 0건이었다. 모델링 데이터의 order_total과 order_items 재계산 값의 최대 차이도 0이었다.
- orders → customers 관계 검증 결과: customers 150행의 customer_id 중복은 0건이었고, customers에 없는 customer_id를 가진 주문은 0건이었다. 병합 후 행 수는 300행으로 orders와 모델링 데이터의 300행과 같았다.

### 나의 해석과 판단
관계 검증 없이 병합만 수행하면 행 수가 조용히 바뀌는 문제가 생길 수 있다. customers의 customer_id가 중복이면 병합 후 주문 행이 복제되어 300행을 넘고, 복제된 행에서 order_items를 합산하면 order_total이 배수로 부풀려진다. order_items를 order_id별로 합산하지 않고 그대로 병합하면 한 주문이 상품 수만큼 여러 행이 되어 764행이 되고, 예측 단위가 주문 1건이라는 정의가 깨진다. 이 경우 상품 수가 많은 주문이 평가에서 더 큰 비중을 갖게 되어 MAE와 RMSE가 왜곡된다.
매칭되지 않는 customer_id가 있으면 age, gender, city가 결측이 되는데, 이 결측이 눈에 띄지 않은 채 모델에 들어갈 수 있다. line_total을 검증하지 않으면 계산 오류나 결측이 Target에 그대로 반영되어 모델이 잘못된 정답을 학습한다.
이번 검증에서는 세 관계가 모두 문제없이 확인되었다. line_total은 수량과 단가의 곱과 전부 일치했고, 주문과 주문 상세는 300건씩 빠짐없이 1대1로 대응했으며, 모든 주문이 customers에 매칭되고 병합 후에도 300행이 유지되었다. 따라서 모델링 데이터의 order_total은 주문 단위로 정확하게 만들어졌고, 이후 결과의 낮은 예측력은 Target 생성이나 병합 오류가 아니라 입력 변수의 설명력 한계에서 비롯된 것으로 해석헀다.

---

## 4. Train / Final Test 분할

- Train 시작일: 2025-09-18
- Train 종료일: 2026-06-18
- Train 행 수: 240
- Final Test 시작일: 2026-06-18
- Final Test 종료일: 2026-09-17
- Final Test 행 수: 60
- 같은 달력 날짜 중복 여부: 있음 (2026-06-18 하루, 훈련 1건과 테스트 2건)

### 분할 판단
이번 문제는 과거 주문으로 이후 주문 금액을 예측하는 상황이므로 시간 순서 분할을 사용했다. 랜덤 분할을 쓰면 테스트 기간 주문보다 늦은 주문이 훈련에 섞여 미래 정보로 과거를 맞히는 구조가 되고, 성능이 실제보다 좋게 나올 수 있다. 시간 순서 분할은 훈련 기간이 테스트 기간보다 앞서도록 해서 실제 서비스에서 예측하는 순서를 그대로 모방했다. 훈련 240건과 테스트 60건의 합은 300건으로 모델링 데이터의 행 수와 같다.
다만 2026-06-18에 훈련 1건과 테스트 2건이 나뉘어 들어가 경계가 날짜 단위로 깨끗하게 분리되지 않았다. 분할 함수에 test_size=0.2를 지정했고 결과가 정확히 80퍼센트와 20퍼센트이므로, 날짜가 아니라 행 비율 지점에서 잘리면서 같은 날 주문이 양쪽에 걸린 것으로 보인다. 훈련 최대 날짜와 테스트 최소 날짜가 같은 날이라 훈련이 테스트보다 늦은 날짜를 갖지는 않았고, 이 3건은 전체 300건의 1퍼센트라 성능에 미치는 영향은 작다고 판단한다. 그래도 엄밀한 날짜 기준 분할은 아니라는 점을 한계로 기록한다.
또 테스트 기간에는 훈련 기간에 없던 7월과 8월이 포함되어 order_month 분포가 어긋나고, 테스트가 60건뿐이라 성능 지표의 변동이 클 수 있다.

![시간 분할](images/step04_split.png)

---
## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline Mean | 414459 | 36686 | -0.026 | 별도 셀로 계산한 기준선이다. 표준편차는 노트북과 같은 모집단 기준(ddof=0)이다 |
| Linear Regression | 486207 | 86093 | -0.530 | 표준편차가 커서 구간별 성능이 불안정하다 |
| Random Forest | 424718 | 10917 | -0.123 | 비베이스라인 중 평균과 표준편차가 모두 가장 낮다 |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest
- 선택 기준: 비베이스라인 모델 중 CV MAE 평균이 가장 낮고(424718 대 486207), CV MAE 표준편차도 가장 작다(10917 대 86093). 다만 Baseline Mean(414459)보다는 평균 오차가 약 2.5퍼센트 높아서, 선택된 모델이 기준선보다 낫다는 뜻은 아니다.
- Final Test 결과를 보기 전에 선택했는가: 아니오

### 나의 해석과 판단
CV 평균은 여러 시점에서 낸 오차의 중심값이고, 표준편차는 그 오차가 기간에 따라 얼마나 흔들리는지를 보여 준다. 평균이 비슷해도 표준편차가 크면 어떤 기간에는 크게 틀린다는 뜻이라 새로운 기간에서의 성능을 믿기 어렵다. 이번 결과에서 Linear Regression은 표준편차가 86093으로 Random Forest의 약 8배였고 평균 오차도 더 높았다. 그래서 두 후보 중에서는 평균과 안정성이 모두 앞선 Random Forest를 선택했다.
그러나 Baseline Mean과 비교하면 선택된 모델의 의미가 제한적이다. Random Forest의 CV MAE 평균은 424718로 Baseline의 414459보다 약 10260 높고, CV R² 평균도 -0.123으로 Baseline의 -0.026보다 낮다. 다만 이 차이는 Baseline의 표준편차 약 36686보다 작아서 Random Forest가 기준선보다 나쁘다고 단정하기는 어렵고, 기준선을 넘지 못했으며 사실상 같은 수준일 수 있다고 해석한다. Linear Regression은 평균이 기준선보다 약 17퍼센트 높아 명확하게 불리했다. 따라서 현재 입력 변수로는 평균 예측을 안정적으로 넘는 모델을 확보하지 못했다.
각 fold의 검증 구간은 훈련 240건을 5분할한 계산상 약 40건이라 표준편차 자체도 불확실하므로 경향 수준으로 판단했다. 또 6절의 Final Test 비교표를 본 뒤에 CV 결과를 확인했기 때문에 이 선택이 Final Test와 완전히 독립적이라고는 할 수 없어서 선택 여부 칸에 "아니오"로 기록했다. 참고로 Final Test MAE는 Linear Regression(480839)이 Random Forest(481964)보다 약 0.2퍼센트 낮아 CV와 순위가 반대지만, 차이가 작고 테스트가 60건뿐이라 선택을 바꿀 근거로 보지 않았다. 두 기준 모두에서 Baseline이 가장 낮은 MAE를 보인 점은 같은 방향이다. 

---

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 412812 | 455896 | 531016 | -0.002 | 0.00% |
| Frozen Selected Model (Random Forest) | selected_by_train_cv | 326284 | 481964 | 560736 | -0.118 | -5.72% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
- Frozen Selected Model은 5번 항목에서 Train CV로 고른 Random Forest이며, Train 데이터로만 학습했고 Final Test로 재학습하거나 튜닝하지 않았다.
- Test MAE는 Frozen Model이 481964, Baseline이 455896으로 Frozen Model이 약 26068 높다. 개선율은 -5.72%다.
- Test RMSE는 560736 대 531016으로 Frozen Model이 약 29720 높고, Test R²는 -0.118 대 -0.002다.
- Train MAE와 Test MAE의 차이는 Frozen Model이 155680, Baseline이 43084로 Frozen Model이 약 3.6배 크다.
- 5번 항목의 CV MAE 평균(424718)보다 Final Test MAE(481964)가 약 13.5퍼센트 높다.
- 사진에는 참고용으로 Linear Regression 행(Test MAE 480839, 개선율 -5.47%)도 함께 보이며, 결과는 같은 방향이다. 

### 나의 해석과 판단
- Frozen Model이 baseline보다 실제로 개선되었는가?
  - 개선되지 않았다. MAE, RMSE, R² 세 지표 모두 Baseline보다 나쁘다. 5번 CV에서도 Random Forest(424718)가 Baseline(414459)보다 높았으므로 같은 방향이다. 다만 Final Test가 60건뿐이라 차이의 크기는 경향 수준으로 해석했다. 또 노트북 실행 순서상 Final Test 표를 CV보다 먼저 확인했으므로 모델 선택이 Final Test와 완전히 독립적이지는 않다.
- MAE와 RMSE 차이에서 무엇을 볼 수 있는가?
  - Frozen Model의 RMSE(560736)는 MAE(481964)보다 약 16퍼센트 크다. RMSE는 큰 오차에 더 큰 벌점을 주므로 큰 오차가 섞여 있다는 뜻이다. 그러나 Baseline도 RMSE가 MAE보다 약 16.5퍼센트 크다(531016 대 455896). 큰 오차의 비중은 모델 때문에 생긴 것이 아니라 주문 금액 자체의 분산이 크기 때문이다. 오차 상위 사례에서도 실제 금액이 낮은 주문을 높게, 높은 주문을 낮게 예측하는 극단값 오차가 두드러진다.
- R²는 어떻게 해석할 수 있는가?
  - Test R²가 -0.118로 음수여서 테스트 기간 평균으로 예측하는 것보다 설명력이 낮다. Baseline의 R²가 -0.002로 0에 가까운 것은 훈련 평균이 테스트 평균과 거의 같기 때문이다. Frozen Model은 입력 6개로 주문 금액의 변동을 설명하지 못했고, 훈련 데이터에만 맞춰진 패턴이 테스트에서 손해로 이어진 것으로 보인다. 앞에서 누수 징후가 없었음을 확인했으므로 낮은 R²는 누수가 아니라 입력 변수의 설명력 한계로 해석했다.

### 업무와 분석적 의미
Frozen Model의 평균 절대 오차는 481964로, 3번 항목에서 확인한 검증 구간별 평균 주문 금액(약 83만에서 96만)의 절반 안팎이다. 이 정도 오차로는 개별 주문 금액을 근거로 한 재고, 매출, 마케팅 의사결정을 하기 어렵다. 더 중요한 점은 단순 평균 예측이 이 모델보다 오차가 작다는 것이다. 복잡한 모델을 운영하면 비용만 늘고 정확도는 오히려 낮아지므로, 현재 입력 변수로는 이 모델을 도입할 이유가 없다.
이 결과는 알고리즘의 문제가 아니라 입력 정보의 한계를 알려 준다. 주문 금액은 상품의 수량과 단가로 결정되는데 예측 시점 설정상 그 정보를 쓸 수 없고, 남은 입력(월, 요일, 나이, 성별, 도시, 결제 수단)은 금액과 거의 관계가 없었다. 개선하려면 모델을 바꾸기보다 예측 시점 이전에 알 수 있는 정보를 늘려야 한다. 4번 항목의 구매 이력 피처가 후보이지만 이력이 있는 주문이 58퍼센트이고 대부분 과거 주문이 1건에서 2건이라 효과는 제한적일 수 있다. 예측 시점을 장바구니 담기 이후로 옮기면 목적이 달라지므로, 실제 서비스에서 어느 시점에 어떤 결정을 내릴지부터 정해야 한다. 기준선을 넘지 못한 모델을 쓰지 않는다는 판단 자체가 유효한 분석 결과다.

---

## 7. 대표 오차 사례

진단 모델은 노트북의 select_diagnostic_model이 테스트 MAE가 가장 낮은 비베이스라인 모델로 고른 Linear Regression이다. 6번의 Frozen Model(Random Forest)과 다르므로 아래 사례는 Linear Regression 기준이다.

### 사례 1
- 실제값: 1877000 (order_id 51, 2026-08-25)
- 예측값: 606400
- residual: 1270600 (실제값 - 예측값, 양수이므로 실제값을 낮게 예측했다)
- absolute error: 1270600 (테스트 60건 중 최대 오차)
- 관찰: 실제 금액이 매우 높은 주문을 약 60만으로 낮게 예측했다. 절대 오차 1270600은 테스트 평균 절대 오차(약 480839)의 약 2.6배이고 중앙값(약 472014)의 약 2.7배다. 오차 상위 10건 중 5건(order_id 51, 96, 183, 21, 15)이 이처럼 실제 금액이 1677000 이상인 고액 주문이고, 이들의 예측은 약 606000에서 885000이다. 주문 상세를 확인하니 상품 3종(985000, 444000, 448000)의 합이 1877000이었고, 단가 197000인 상품 하나가 전체의 약 52퍼센트를 차지했다.
- 가능한 이유 후보: 입력 6개(월, 요일, 나이, 성별, 도시, 결제 수단)에는 상품의 수량과 단가 정보가 없어 고액 주문을 구분할 단서가 없다. 이 주문은 고가 상품 하나와 여러 상품 구매가 겹친 경우라 더욱 그렇다. 8월 주문이라 훈련 기간에 없던 월이어서 order_month 값이 외삽되었을 가능성도 있다.
- 추가 확인: 고액 주문이 특정 결제 수단, 도시, 연령대에 몰리는지 확인한다. 나머지 고액 주문 4건(96, 183, 21, 15)도 고가 상품 때문인지 상품 수 때문인지 확인한다.

### 사례 2
- 실제값: 10000 (order_id 168, 2026-08-21)
- 예측값: 980271
- residual: -970271 (실제값 - 예측값, 음수이므로 실제값을 높게 예측했다)
- absolute error: 970271
- 관찰: 실제 금액이 매우 낮은 주문을 약 98만으로 높게 예측했다. 주문 상세를 확인하니 상품 1종, 수량 2, 단가 5000으로 line_total 합이 10000과 일치해서 입력 오류가 아닌 정상 소액 주문이다. 같은 방향의 오차가 order_id 179(실제 86000, 예측 약 992975), 73(실제 72000, 예측 약 972120), 153(실제 172000, 예측 약 1034316), 248(실제 55000, 예측 약 911134)에서도 나타나 상위 10건 중 5건이 저액 주문이다. 저액 주문 5건의 예측은 약 911000에서 1034000이고 고액 주문 5건의 예측(약 606000에서 885000)보다 높아서, 실제 금액 차이를 거의 반영하지 못하고 중간 수준 근처로 예측했다. 상위 10건 중 8건이 2026-07 이후 주문이고, 나머지 2건(153, 248)은 훈련과 테스트가 나뉜 경계일 2026-06-18의 테스트 주문이다.
- 가능한 이유 후보: 이 주문은 상품 1종에 수량 2, 단가 5000이어서 주문 상세 규모 자체가 매우 작다. 이런 규모를 구분할 정보가 입력에 없고, 선형 회귀는 입력 조합에서 나온 평균적인 수준을 예측하므로 저액 주문을 별도로 식별하지 못했을 가능성이 있다. 훈련에 없던 7월과 8월의 월 분포 영향도 후보다.
- 추가 확인: 단가 5000은 다른 주문의 단가(사례 1의 111000에서 197000)보다 크게 낮으므로, 저단가 상품이 소액 주문의 공통 원인인지 order_items 전체의 단가 분포로 확인한다. 경계일 주문 153과 248이 둘 다 상위 오차에 든 것이 우연인지도 별도로 확인이 필요하다.

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

---

## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

## 8. Validation Evidence

`reports/ch09_regression_validation.csv`는 이 프로젝트 코드 버전에서 생성되지 않아, 같은 항목을 노트북 셀에서 직접 계산해 작성했다.

| check | value | status |
| --- | --- | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | False | FAIL |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 60 | PASS |

### FAIL 항목
- 없음 / 있음: 있음 
- 있다면 원인과 수정 내용: 분할이 날짜가 아니라 행 비율(test_size=0.2) 지점에서 이루어져 2026-06-18에 훈련 1건과 테스트 2건이 걸쳤다. 훈련 최대 날짜와 테스트 최소 날짜가 같아 엄격한 부등식이 성립하지 않는다. 미래 주문이 훈련에 섞인 것이 아니라 같은 날짜가 양쪽에 걸린 경우다. 주어진 분할 함수는 바꾸지 않았고, 해당 3건이 전체 300건의 1퍼센트라 영향이 작다고 판단해 4번 항목에 한계로 기록했다.

---

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
order_id가 포함된 예측 결과는 개별 주문 단위의 기록이라 Internal로 구분했다. ch09_regression_predictions_internal.csv에는 order_id, order_date, 실제 주문 금액, 예측 금액, 잔차가 한 행에 함께 들어 있다. order_id는 주문 시스템의 원본 데이터와 연결하는 키라서, 이 파일이 외부에 나가면 받은 사람이 주문 날짜와 금액으로 특정 주문을 찾고, 같은 키로 customers와 orders의 다른 정보까지 이어 붙일 수 있다. ch09_regression_model_data_internal.csv는 order_id에 더해 나이, 성별, 도시, 결제 수단이 주문 단위로 붙어 있어 재식별 위험이 더 크다.
반면 model_comparison, cv_summary, split_summary, feature_audit, validation은 모델 단위나 항목 단위로 집계한 값과 컬럼 이름만 담고 있어 개별 주문을 가리키지 않는다. report.md와 그림 두 장도 요약 지표와 분포만 보여 주므로 공개 가능한 결과로 분리했다.
이 구분은 파일 이름의 internal 표기로 드러나므로, 제출이나 공유 과정에서 실수로 섞이는 일을 줄여 준다. 또 오차 상위 사례처럼 개별 주문을 설명하는 내용은 분석에는 필요하지만 공개 문서에는 order_id를 그대로 쓰지 않고, 필요하면 주문 A, 주문 B처럼 바꿔 쓰는 것이 일관된 기준이다.

---

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Final Test에서 선택 모델(Random Forest)이 단순 평균 Baseline을 넘지 못했다. Test MAE는 481964 대 455896으로 개선율이 -5.72%이고, Test RMSE는 560736 대 531016, Test R²는 -0.118 대 -0.002다. 5번 Train CV에서도 Random Forest의 MAE 평균(424718)이 Baseline(414459)보다 높아 같은 방향이다. 기준선보다 나은 모델이 없으므로 이 모델을 쓸 이유가 없다.
2. 낮은 성능의 원인은 누수나 데이터 생성 오류가 아니라 입력 변수의 설명력 부족으로 판단헀다. 2번 점검에서 사용 입력 6개와 금지 입력 10개가 겹치지 않았고, 3번에서 order_total이 order_items 합산과 최대 차이 0으로 일치했다. 숫자형 입력과 order_total의 상관도 절댓값 0.089 이하였다. 예측 시점에 알 수 있는 정보(월, 요일, 나이, 성별, 도시, 결제 수단)만으로는 주문 금액을 구분할 단서가 없다.
3. 평균 예측보다 못한 모델을 운영하면 비용만 늘고 정확도는 오히려 낮아진다. 참고용으로 쓰더라도 사용자가 이 수치를 근거로 판단할 수 있어서, 개선 전에는 사용하지 않는 것이 안전하다.

### 업무적 위험
- 오차가 양방향으로 극단적이다. 7번 사례에서 실제 1877000인 주문을 606400으로, 실제 10000인 주문을 980271로 예측했다. 고액 주문을 저평가하고 소액 주문을 고평가하므로 재고, 매출 추정, 마케팅 대상 선정에 쓰면 판단을 거꾸로 이끌 수 있다.
- 평균 절대 오차 481964는 과제 3의 검증 구간 평균 주문 금액(약 83만에서 96만)의 절반 안팎이다. 개별 주문 단위의 금액 예측으로는 쓸 수 없는 수준이다.
- 모델이 그럴듯한 숫자를 출력하기 때문에 R²가 음수라는 사실을 모르는 사용자는 예측값을 신뢰할 수 있다. 사용 보류 결정을 보고서에 명시해야 한다.
- 예측 결과와 모델링 데이터에는 order_id와 고객 속성이 들어 있어 Internal로 구분했다. 이 파일들이 외부에 나가면 개별 주문을 재식별할 위험이 있다.

### 현재 데이터의 한계
- 표본이 작다. 주문 300건, 고객 150명이고 Final Test는 60건뿐이라 지표의 변동이 크다. CV의 각 검증 구간도 계산상 약 40건이다.
- 계절 범위가 어긋난다. 훈련 기간에는 7월과 8월이 없고 테스트 기간에만 나타나 order_month 분포가 어긋난다. 오차 상위 10건 중 8건이 2026-07 이후 주문이었다.
- 분할 경계가 깨끗하지 않다. 8번에서 strict_train_before_test가 FAIL이었고, 2026-06-18에 훈련 1건과 테스트 2건이 걸쳤다. 영향은 작다고 판단하지만 엄밀한 날짜 분할은 아니다.
- 모델 선택이 Final Test와 완전히 독립적이지 않다. 노트북 순서상 Final Test 표를 CV보다 먼저 확인했다.
- 예측 시점 가정이 교육용이다. 결제 수단 선택 시점에 장바구니가 이미 확정된다면 payment_method는 사후 정보가 된다.
- 구매 이력 정보가 없다. 4번 항목에서 이력이 있는 주문은 58퍼센트이고 대부분 과거 주문이 1건에서 2건이라 이력 피처의 효과도 제한적일 수 있다.

### 다음 개선 우선순위
1. 예측 시점 이전에 알 수 있는 정보를 늘린다. 4번 항목의 prev_order_count, prev_avg_total, days_since_last_order를 shift(1) 기준으로 추가하고, 집계 기준일을 명시해 미래 정보가 섞이지 않게 한다. 먼저 Baseline 대비 개선이 생기는지 같은 분할과 파이프라인에서 비교한다.
2. 데이터와 분할을 보강한다. 1년 이상의 주문 기간을 확보해 7월과 8월이 훈련에도 포함되게 하고, 분할을 날짜 경계로 다시 해서 같은 날짜가 양쪽에 걸리지 않게 한다. 반복 시드와 더 많은 fold로 성능의 일관성도 확인한다.
3. 문제 정의를 바꾼다. 금액 자체를 맞히기 어렵다면 고액 주문 여부 같은 분류 문제로 바꾸거나, 사용 목적에 맞는 예측 시점을 정한다. 모델 복잡도 조정은 우선순위가 낮다. 2번 항목에서 min_samples_leaf 20이 테스트 MAE 464494로 가장 낮았지만 이는 별도 파이프라인 값이라 Baseline과 같은 조건에서 다시 비교해야 한다.
---

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: 성공
- Train CV Selected Model: Random Forest. 교차검증 표에서 비베이스라인 모델 중 평균과 표준편차가 모두 가장 낮아 선택했다. 다만 Baseline Mean의 CV MAE 평균(414459)보다 높아 기준선을 넘지는 못했다. 스크립트 출력의 진단 모델(Linear Regression)은 테스트 MAE 기준의 진단용 선택이라 이 항목과 구분한다.
- Final Validation 전체 PASS 여부: 아니오. 6개 항목 중 5개는 PASS이고 strict_train_before_test만 FAIL이다. 2026-06-18에 훈련 1건과 테스트 2건이 걸쳐 훈련 최대일과 테스트 최소일이 같은 날이 되었기 때문이다. 이 판정은 스크립트가 만든 파일이 아니라 노트북 셀로 직접 만든 validation 파일 기준이다.
- Notebook 결과와 스크립트 결과의 일관성: 일치한다. 모델링 데이터 (300, 11), 훈련 240행과 테스트 60행, 훈련과 테스트 기간, 모델 비교표(Test MAE 455896, 480839, 481964, Test R² -0.002, -0.128, -0.118), 교차검증 표, 진단 모델, 오차 상위 10건이 노트북 출력과 같다. 일치를 확인한 범위는 출력된 값이며 저장 파일 전체의 비교는 아니다.

---

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.